In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import cv2
import numpy as np
import torch
import gc
import time
import torch
from realesrgan.archs.rrdb_unet_v4_t_arch import RRDB_UNet_v4_t
from realesrgan.archs.rrdb_unet_v5_t_arch import RRDB_UNet_v5_t

In [ ]:
pad = 8
device = "cuda"
#device = "cpu"

In [ ]:
model = RRDB_UNet_v5_t(
    num_in_ch=6,
    num_out_ch=3,
    highway_channels_base=32,
    processing_channels_base=16,
    num_grow_ch_base = 8,
    ae_rrdb_blocks=4,
    ae_channel_multipliers = [1,2,4,8,16],
    use_attention=True,
    body_rrdb_blocks = 6,
    res1_add=False,
    inference=True,
    memory_efficient_inference_device = "cuda"
)

model = RRDB_UNet_v4_t(
    num_in_ch=6,
    num_out_ch=3,
    highway_channels_base=32,
    processing_channels_base=16,
    num_grow_ch_base = 8,
    ae_rrdb_blocks=6,
    ae_channel_multipliers = [1,4,8,16,24],
    use_attention=True,
    body_rrdb_blocks = 10,
    res1_add=False,
    inference=True,
    memory_efficient_inference_device = device
)

In [ ]:
img = cv2.imread("tests/data/lq_4/comic.png")
#img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#img = cv2.resize(img, (4020*1, 4020*4))
#img = cv2.resize(img, (4020*1, 4020*2))
scale = 4
img = cv2.resize(img, (int(img.shape[1]*scale),int(img.shape[0]*scale)))
print(img.shape)
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

img = img + torch.randn_like(torch.tensor(img).float()).numpy() * 0.05 * 255
img = np.array(np.clip(img, 0, 255), dtype="uint8")
plt.imshow(cv2.cvtColor(np.clip(img,0,255), cv2.COLOR_BGR2RGB))

In [ ]:
# Note: pytorch appears to use different gpu code if you exceed some resolution causing it to be very slow.
# compiling with fixed resolution makes it fast again, but it requires bucketing.
# see esrgan_arch_test2

In [ ]:
from realesrgan.real_srflow import RealSRFLOW

#srflow = RealSRFLOW(None, model, pad, device, 3)
srflow = RealSRFLOW("experiments/train_flow_v2/models/net_g_160000.pth", model, pad, device)

In [ ]:
out = srflow.enhance(img, 5)

In [ ]:
import matplotlib.pyplot as plt

for i in out:
    plt.imshow(cv2.cvtColor(i, cv2.COLOR_BGR2RGB))
    plt.show()

In [ ]:
#exit()